# Aufgabe 4c: CellCNN

Dieses Notebook implementiert CellCNN klein und nachvollziehbar mit aktuellem PyTorch. Es orientiert sich am Paper und an der offiziellen Referenz: Ein linearer Filter wird auf jede Zelle angewendet, eine ReLU erzeugt nichtnegative Zellantworten, die stärksten 1 % werden je Filter gemittelt und eine lineare Softmax-Ausgabe sagt das Spenderlabel vorher.

Der voreingestellte Smoke-Modus reduziert Multi-Cell-Inputs und Epochen und verwendet `gated_NK`. Diese Resultate sind nicht für den Bericht bestimmt. Der Full-Modus verwendet `gated_alive` und die papernahen Parameter.

## 1. Konfiguration

In [1]:
from pathlib import Path
import copy
import csv
import os
import warnings

import flowkit as fk
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import auc, balanced_accuracy_score, precision_recall_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RUN_MODE = os.environ.get("TASK4_RUN_MODE", "smoke")
RUN_TRAINING = os.environ.get("TASK4_RUN_TRAINING", "1") == "1"
COFACTOR = 5.0
LEARNING_RATE = 0.01
TOP_FRACTION = 0.01
L2_COEFFICIENT = 1e-4
IMPLEMENTATION_VERSION = "pytorch_materialized_v1"

if RUN_MODE == "smoke":
    GATE = "gated_NK"
    SPLIT_IDS = [0, 1]
    TRAINING_CELLS_PER_INPUT = 1_000
    TRAINING_INPUTS_PER_DONOR = 8
    PREDICTION_CELLS_PER_INPUT = 3_000
    PREDICTION_INPUTS_PER_DONOR = 5
    SCALER_CELLS_PER_DONOR = 2_000
    FILTER_COUNTS = [3, 5]
    BATCH_SIZE = 16
    MAX_EPOCHS = 12
    EARLY_STOPPING_PATIENCE = 3
elif RUN_MODE == "full":
    GATE = "gated_alive"
    SPLIT_IDS = list(range(100))
    TRAINING_CELLS_PER_INPUT = 3_000
    TRAINING_INPUTS_PER_DONOR = 200
    PREDICTION_CELLS_PER_INPUT = 20_000
    PREDICTION_INPUTS_PER_DONOR = 5
    SCALER_CELLS_PER_DONOR = 20_000
    FILTER_COUNTS = [3, 4, 5]
    BATCH_SIZE = 128
    MAX_EPOCHS = 100
    EARLY_STOPPING_PATIENCE = 5
else:
    raise ValueError(f"Unbekannter RUN_MODE: {RUN_MODE}")
SPLIT_LIMIT = int(os.environ.get("TASK4_SPLIT_LIMIT", "0"))
if SPLIT_LIMIT > 0:
    SPLIT_IDS = SPLIT_IDS[:SPLIT_LIMIT]

GATE_SUFFIXES = {"gated_NK": "_NK", "gated_alive": "_alive"}
EXPECTED_EVENT_COUNTS = {"gated_NK": 261_593, "gated_alive": 3_438_750}
GATE_SUFFIX = GATE_SUFFIXES[GATE]

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "NK_cell_dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "NK_cell_dataset" / "NK_cell_dataset"
FCS_DIR = DATA_ROOT / "NK_cell_dataset" / GATE
LABELS_PATH = DATA_ROOT / "NK_fcs_samples_with_labels.csv"
MARKERS_PATH = DATA_ROOT / "NK_markers.csv"
SPLITS_PATH = PROJECT_ROOT / "results" / "tables" / "task4_donor_splits.csv"
PREDICTIONS_PATH = (
    PROJECT_ROOT / "results" / "tables"
    / f"task4_cellcnn_predictions_{GATE}_{RUN_MODE}.csv"
)
SELECTION_PATH = (
    PROJECT_ROOT / "results" / "tables"
    / f"task4_cellcnn_selection_{GATE}_{RUN_MODE}.csv"
)
FILTERS_PATH = (
    PROJECT_ROOT / "results" / "tables"
    / f"task4_cellcnn_filters_{GATE}_{RUN_MODE}.csv"
)

for required_path in (FCS_DIR, LABELS_PATH, MARKERS_PATH, SPLITS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Erwarteter Pfad fehlt: {required_path}")

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(0)
print(
    f"PyTorch: {torch.__version__}; Modus: {RUN_MODE}; Gate: {GATE}; "
    f"Gerät: {DEVICE}; Training: {'aktiv' if RUN_TRAINING else 'deaktiviert'}"
)

PyTorch: 2.14.0+cu130; Modus: smoke; Gate: gated_NK; Gerät: cuda; Training: deaktiviert


## 2. Daten und gemeinsame Splits laden

Die FCS-Dateien werden read-only geladen und mit `arcsinh(x / 5)` transformiert. Alle Arrays bleiben spenderweise getrennt.

In [2]:
with MARKERS_PATH.open(newline="", encoding="utf-8-sig") as stream:
    markers = next(csv.reader(stream))

label_table = pd.read_csv(LABELS_PATH)
label_table["donor_id"] = label_table["fcs_filename"].str.replace(
    r"_NK\.fcs$", "", regex=True
)
label_table["label"] = label_table["label"].astype(int)
fcs_paths = sorted(FCS_DIR.glob("*.fcs"))
fcs_table = pd.DataFrame(
    {
        "donor_id": [path.stem.removesuffix(GATE_SUFFIX) for path in fcs_paths],
        "fcs_path": fcs_paths,
    }
)
sample_table = (
    label_table[["donor_id", "label"]]
    .merge(fcs_table, on="donor_id", validate="one_to_one")
    .sort_values("donor_id")
    .reset_index(drop=True)
)
donor_splits = pd.read_csv(SPLITS_PATH)

assert len(markers) == 37
assert len(fcs_paths) == 20
assert len(sample_table) == 20
assert set(SPLIT_IDS).issubset(set(donor_splits["split_id"]))
split_labels = donor_splits[["donor_id", "label"]].drop_duplicates()
assert split_labels.merge(
    sample_table[["donor_id", "label"]],
    on=["donor_id", "label"],
    validate="one_to_one",
).shape[0] == len(sample_table)

def load_transformed_fcs(path: Path) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=r"FCS file .* reported incorrect data offset.*"
        )
        sample = fk.Sample(str(path), ignore_offset_error=True)
    frame = sample.as_dataframe(source="raw")
    short_names = pd.Index(sample.pns_labels, name="marker")
    if not short_names.is_unique:
        raise ValueError(f"Doppelte FCS-Kurznamen in {path.name}.")
    frame.columns = short_names
    missing = [marker for marker in markers if marker not in frame.columns]
    if missing:
        raise ValueError(f"Fehlende Marker in {path.name}: {missing}")
    values = frame.loc[:, markers].to_numpy(dtype=np.float32, copy=True)
    values = np.arcsinh(values / COFACTOR).astype(np.float32, copy=False)
    if not np.isfinite(values).all():
        raise ValueError(f"Nicht-endliche Werte in {path.name}.")
    return values

data_by_donor = {
    row.donor_id: load_transformed_fcs(row.fcs_path)
    for row in sample_table.itertuples(index=False)
}
label_by_donor = sample_table.set_index("donor_id")["label"].to_dict()
assert sum(map(len, data_by_donor.values())) == EXPECTED_EVENT_COUNTS[GATE]

display(
    pd.DataFrame(
        {
            "donor_id": list(data_by_donor),
            "label": [label_by_donor[x] for x in data_by_donor],
            "cells": [len(data_by_donor[x]) for x in data_by_donor],
        }
    )
)

,donor_id,label,cells
0,a_001,1,6320
1,a_002,1,15809
2,a_003,0,11438
3,a_004,0,17443
4,a_005,1,25226
5,a_006,0,5652
6,a_007,1,9819
7,a_009,0,20401
8,a_010,0,9092
9,a_011,0,31424


## 3. CellCNN-Architektur und Multi-Cell-Inputs

Trainingsinputs werden mit Zurücklegen aus jeweils einem Spender gezogen. Jeder Spender liefert gleich viele Inputs und erhält daher dasselbe Gewicht. Die Skalierung wird mit gleich vielen Zellen je innerem Trainingsspender gefittet.

In [3]:
def materialize_multicell_inputs(
    donor_ids: list[str],
    scaled_data: dict[str, np.ndarray],
    cells_per_input: int,
    inputs_per_donor: int,
    seed: int,
) -> TensorDataset:
    donor_ids = sorted(donor_ids)
    input_count = len(donor_ids) * inputs_per_donor
    values = np.empty(
        (input_count, cells_per_input, len(markers)), dtype=np.float32
    )
    labels = np.empty(input_count, dtype=np.int64)
    input_index = 0
    for donor_id in donor_ids:
        donor_values = scaled_data[donor_id]
        for _ in range(inputs_per_donor):
            rng = np.random.default_rng(seed + input_index)
            cell_indices = rng.integers(
                0, len(donor_values), size=cells_per_input
            )
            values[input_index] = donor_values[cell_indices]
            labels[input_index] = label_by_donor[donor_id]
            input_index += 1
    return TensorDataset(
        torch.from_numpy(values).to(DEVICE),
        torch.from_numpy(labels).to(DEVICE),
    )


class CellCNN(nn.Module):
    def __init__(self, marker_count: int, filter_count: int, top_fraction: float):
        super().__init__()
        self.cell_filters = nn.Linear(marker_count, filter_count)
        self.output_layer = nn.Linear(filter_count, 2)
        self.top_fraction = top_fraction

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        cell_responses = torch.relu(self.cell_filters(values))
        top_count = max(1, int(self.top_fraction * values.shape[1]))
        pooled = torch.topk(cell_responses, k=top_count, dim=1).values.mean(dim=1)
        return self.output_layer(pooled)


def fit_balanced_scaler(donor_ids: list[str], cells_per_donor: int, seed: int):
    rng = np.random.default_rng(seed)
    scaler_parts = []
    for donor_id in sorted(donor_ids):
        values = data_by_donor[donor_id]
        if len(values) < cells_per_donor:
            raise ValueError(f"Zu wenige Zellen für Skalierung: {donor_id}")
        indices = rng.choice(len(values), size=cells_per_donor, replace=False)
        scaler_parts.append(values[indices])
    scaler = StandardScaler().fit(np.concatenate(scaler_parts))
    return scaler


def scale_donors(donor_ids: list[str], scaler: StandardScaler):
    return {
        donor_id: scaler.transform(data_by_donor[donor_id]).astype(np.float32, copy=False)
        for donor_id in donor_ids
    }

## 4. Training und Early Stopping

Adam minimiert die Cross-Entropy plus L2-Strafe auf Filter- und Ausgabegewichte. Early Stopping überwacht den Verlust fester Multi-Cell-Inputs der inneren Validierungsspender.

In [4]:
def regularized_loss(model: CellCNN, logits: torch.Tensor, labels: torch.Tensor):
    cross_entropy = nn.functional.cross_entropy(logits, labels)
    weight_penalty = (
        model.cell_filters.weight.square().sum()
        + model.output_layer.weight.square().sum()
    )
    return cross_entropy + L2_COEFFICIENT * weight_penalty


def evaluate_loader(model: CellCNN, loader: DataLoader) -> float:
    model.eval()
    weighted_loss_sum = 0.0
    example_count = 0
    with torch.no_grad():
        for values, labels in loader:
            logits = model(values.to(DEVICE))
            batch_loss = float(regularized_loss(model, logits, labels.to(DEVICE)))
            weighted_loss_sum += batch_loss * len(labels)
            example_count += len(labels)
    return weighted_loss_sum / example_count


def train_candidate(
    train_ids: list[str],
    validation_ids: list[str],
    filter_count: int,
    seed: int,
) -> tuple[CellCNN, StandardScaler, dict[str, object]]:
    torch.manual_seed(seed)
    scaler = fit_balanced_scaler(train_ids, SCALER_CELLS_PER_DONOR, seed)
    scaled_data = scale_donors(train_ids + validation_ids, scaler)
    train_dataset = materialize_multicell_inputs(
        train_ids, scaled_data, TRAINING_CELLS_PER_INPUT,
        TRAINING_INPUTS_PER_DONOR, seed + 1_000,
    )
    validation_dataset = materialize_multicell_inputs(
        validation_ids, scaled_data, TRAINING_CELLS_PER_INPUT,
        TRAINING_INPUTS_PER_DONOR, seed + 2_000,
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, generator=generator,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model = CellCNN(len(markers), filter_count, TOP_FRACTION).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_state = copy.deepcopy(model.state_dict())
    best_validation_loss = np.inf
    epochs_without_improvement = 0
    history = []

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_losses = []
        for values, labels in train_loader:
            optimizer.zero_grad()
            logits = model(values.to(DEVICE))
            loss = regularized_loss(model, logits, labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach()))

        validation_loss = evaluate_loader(model, validation_loader)
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": float(np.mean(train_losses)),
                "validation_loss": validation_loss,
            }
        )
        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                break

    model.load_state_dict(best_state)
    training_info = {
        "epochs_run": len(history),
        "best_validation_loss": best_validation_loss,
        "history": history,
    }
    return model, scaler, training_info

## 5. Spenderweise Vorhersage und innere Modellauswahl

Mehrere zufällige Multi-Cell-Vorhersagen werden zu einer CMV+-Wahrscheinlichkeit pro Spender gemittelt. Dies ist eine bewusste Abweichung vom NK-Benchmark im Paper, der im Test einen einzelnen Input mit 20.000 Zellen verwendet: Die Mittelung von fünf Inputs reduziert die Monte-Carlo-Streuung der Spendervorhersage und entspricht dem vorab festgelegten Projektplan.

Auch die Kandidaten werden anhand dieser spenderweise gemittelten Vorhersagen statt anhand einzelner generierter Validierungsinputs bewertet. Damit bleibt der Spender als unabhängige Beobachtungseinheit maßgeblich und viele Multi-Cell-Inputs desselben Spenders werden nicht als unabhängige Validierungsfälle behandelt. Die Kandidatenauswahl erfolgt wie im Paper primär nach Validierungsgenauigkeit; ROC-AUC und Validierungsverlust lösen Gleichstände.

In [5]:
def predict_donor(
    model: CellCNN,
    scaler: StandardScaler,
    donor_id: str,
    seed: int,
) -> float:
    values = data_by_donor[donor_id]
    input_cell_count = min(PREDICTION_CELLS_PER_INPUT, len(values))
    rng = np.random.default_rng(seed)
    probabilities = []
    model.eval()
    with torch.no_grad():
        for _ in range(PREDICTION_INPUTS_PER_DONOR):
            indices = rng.choice(len(values), size=input_cell_count, replace=False)
            scaled = scaler.transform(values[indices]).astype(np.float32, copy=False)
            logits = model(torch.from_numpy(scaled).unsqueeze(0).to(DEVICE))
            probability = torch.softmax(logits, dim=1)[0, 1].item()
            probabilities.append(probability)
    return float(np.mean(probabilities))


def train_outer_split(split_id: int):
    split = donor_splits.loc[donor_splits["split_id"] == split_id]
    split_seed = int(split["split_seed"].iloc[0])
    test_ids = split.loc[split["outer_partition"] == "test", "donor_id"].tolist()
    candidates = []
    selection_records = []

    for inner_fold in range(3):
        train_ids = split.loc[
            (split["outer_partition"] == "train")
            & (split["inner_fold"] != inner_fold), "donor_id"
        ].tolist()
        validation_ids = split.loc[
            (split["outer_partition"] == "train")
            & (split["inner_fold"] == inner_fold), "donor_id"
        ].tolist()
        assert not set(train_ids) & set(validation_ids)
        assert not set(train_ids + validation_ids) & set(test_ids)

        for filter_count in FILTER_COUNTS:
            candidate_seed = split_seed + 10_000 * inner_fold + 100 * filter_count
            model, scaler, training_info = train_candidate(
                train_ids, validation_ids, filter_count, candidate_seed
            )
            validation_scores = np.array(
                [
                    predict_donor(model, scaler, donor_id, candidate_seed + 50_000 + index)
                    for index, donor_id in enumerate(sorted(validation_ids))
                ]
            )
            validation_true = np.array(
                [label_by_donor[x] for x in sorted(validation_ids)]
            )
            validation_accuracy = float(
                ((validation_scores >= 0.5).astype(int) == validation_true).mean()
            )
            validation_auc = float(roc_auc_score(validation_true, validation_scores))
            record = {
                "split_id": split_id,
                "gate": GATE,
                "run_mode": RUN_MODE,
                "device": DEVICE.type,
                "implementation_version": IMPLEMENTATION_VERSION,
                "inner_fold": inner_fold,
                "filter_count": filter_count,
                "candidate_seed": candidate_seed,
                "validation_accuracy": validation_accuracy,
                "validation_roc_auc": validation_auc,
                "best_validation_loss": training_info["best_validation_loss"],
                "epochs_run": training_info["epochs_run"],
            }
            selection_records.append(record)
            candidates.append({"model": model, "scaler": scaler, **record})

    candidates.sort(
        key=lambda item: (
            -item["validation_accuracy"],
            -item["validation_roc_auc"],
            item["best_validation_loss"],
            item["filter_count"],
            item["inner_fold"],
        )
    )
    selected = candidates[0]
    for record in selection_records:
        record["selected"] = (
            record["inner_fold"] == selected["inner_fold"]
            and record["filter_count"] == selected["filter_count"]
        )

    prediction_records = []
    for index, donor_id in enumerate(sorted(test_ids)):
        score = predict_donor(
            selected["model"], selected["scaler"], donor_id,
            split_seed + 900_000 + index,
        )
        prediction_records.append(
            {
                "method": "cellcnn",
                "gate": GATE,
                "run_mode": RUN_MODE,
                "device": DEVICE.type,
                "implementation_version": IMPLEMENTATION_VERSION,
                "split_id": split_id,
                "split_seed": split_seed,
                "donor_id": donor_id,
                "y_true": label_by_donor[donor_id],
                "score": score,
                "decision_threshold": 0.5,
                "y_pred": int(score >= 0.5),
                "filter_count": selected["filter_count"],
                "selected_inner_fold": selected["inner_fold"],
                "training_cells_per_input": TRAINING_CELLS_PER_INPUT,
                "training_inputs_per_donor": TRAINING_INPUTS_PER_DONOR,
                "prediction_cells_per_input": min(
                    PREDICTION_CELLS_PER_INPUT, len(data_by_donor[donor_id])
                ),
                "prediction_inputs_per_donor": PREDICTION_INPUTS_PER_DONOR,
                "top_fraction": TOP_FRACTION,
            }
        )

    filter_records = []
    filter_weights = selected["model"].cell_filters.weight.detach().cpu().numpy()
    filter_biases = selected["model"].cell_filters.bias.detach().cpu().numpy()
    output_weights = selected["model"].output_layer.weight.detach().cpu().numpy()
    output_contrasts = output_weights[1] - output_weights[0]
    for filter_id in range(selected["filter_count"]):
        for marker_index, marker in enumerate(markers):
            filter_records.append(
                {
                    "split_id": split_id,
                    "split_seed": split_seed,
                    "gate": GATE,
                    "run_mode": RUN_MODE,
                    "device": DEVICE.type,
                    "implementation_version": IMPLEMENTATION_VERSION,
                    "cofactor": COFACTOR,
                    "top_fraction": TOP_FRACTION,
                    "inner_fold": selected["inner_fold"],
                    "filter_id": filter_id,
                    "marker": marker,
                    "filter_weight": float(filter_weights[filter_id, marker_index]),
                    "filter_bias": float(filter_biases[filter_id]),
                    "output_weight_contrast": float(output_contrasts[filter_id]),
                    "scaler_mean": float(selected["scaler"].mean_[marker_index]),
                    "scaler_scale": float(selected["scaler"].scale_[marker_index]),
                }
            )

    return (
        pd.DataFrame(prediction_records),
        pd.DataFrame(selection_records),
        pd.DataFrame(filter_records),
    )

## 6. CellCNN ausführen oder vorhandene Ergebnisse laden

In [6]:
predictions_by_split = []
selections_by_split = []
filters_by_split = []
completed_split_ids = set()
if (
    RUN_TRAINING
    and PREDICTIONS_PATH.exists()
    and SELECTION_PATH.exists()
    and FILTERS_PATH.exists()
):
    previous_predictions = pd.read_csv(PREDICTIONS_PATH)
    previous_selection = pd.read_csv(SELECTION_PATH)
    previous_filters = pd.read_csv(FILTERS_PATH)
    if (
        "device" in previous_predictions.columns
        and "implementation_version" in previous_predictions.columns
        and previous_predictions["gate"].eq(GATE).all()
        and previous_predictions["run_mode"].eq(RUN_MODE).all()
        and previous_predictions["device"].eq(DEVICE.type).all()
        and previous_predictions["implementation_version"].eq(IMPLEMENTATION_VERSION).all()
    ):
        predictions_by_split.append(previous_predictions)
        selections_by_split.append(previous_selection)
        filters_by_split.append(previous_filters)
        completed_split_ids = set(previous_predictions["split_id"])
        print(f"Vorhandene CellCNN-Ergebnisse geladen: {len(completed_split_ids)} Splits.")

if RUN_TRAINING:
    for split_id in SPLIT_IDS:
        if split_id in completed_split_ids:
            continue
        predictions, selection, learned_filters = train_outer_split(split_id)
        predictions_by_split.append(predictions)
        selections_by_split.append(selection)
        filters_by_split.append(learned_filters)
        cellcnn_predictions = pd.concat(predictions_by_split, ignore_index=True)
        cellcnn_selection = pd.concat(selections_by_split, ignore_index=True)
        cellcnn_filters = pd.concat(filters_by_split, ignore_index=True)
        PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
        cellcnn_predictions.to_csv(PREDICTIONS_PATH, index=False)
        cellcnn_selection.to_csv(SELECTION_PATH, index=False)
        cellcnn_filters.to_csv(FILTERS_PATH, index=False)
        print(f"Split {split_id} abgeschlossen.")
else:
    print("Training deaktiviert; vorhandene CellCNN-Ergebnisse werden geladen.")
    predictions_by_split = [pd.read_csv(PREDICTIONS_PATH)]
    selections_by_split = [pd.read_csv(SELECTION_PATH)]
    filters_by_split = [pd.read_csv(FILTERS_PATH)]

cellcnn_predictions = pd.concat(predictions_by_split, ignore_index=True)
cellcnn_selection = pd.concat(selections_by_split, ignore_index=True)
cellcnn_filters = pd.concat(filters_by_split, ignore_index=True)

assert set(cellcnn_predictions["split_id"]) == set(SPLIT_IDS)
assert cellcnn_predictions.groupby(["split_id", "donor_id"]).size().eq(1).all()
assert cellcnn_predictions.groupby("split_id").size().eq(6).all()
assert cellcnn_predictions["gate"].eq(GATE).all()
assert cellcnn_predictions["run_mode"].eq(RUN_MODE).all()
assert set(cellcnn_predictions["device"]).issubset({"cpu", "cuda"})
if RUN_TRAINING:
    assert cellcnn_predictions["device"].eq(DEVICE.type).all()
assert cellcnn_predictions["implementation_version"].eq(IMPLEMENTATION_VERSION).all()
assert cellcnn_predictions["training_cells_per_input"].eq(TRAINING_CELLS_PER_INPUT).all()
assert cellcnn_predictions["training_inputs_per_donor"].eq(TRAINING_INPUTS_PER_DONOR).all()
assert cellcnn_predictions["prediction_inputs_per_donor"].eq(PREDICTION_INPUTS_PER_DONOR).all()
assert cellcnn_predictions["top_fraction"].eq(TOP_FRACTION).all()
assert np.isfinite(cellcnn_predictions["score"]).all()
assert cellcnn_predictions["score"].between(0, 1).all()
expected_test_rows = donor_splits.loc[
    donor_splits["split_id"].isin(SPLIT_IDS)
    & donor_splits["outer_partition"].eq("test"),
    ["split_id", "donor_id", "label"],
].rename(columns={"label": "y_true"})
observed_test_rows = cellcnn_predictions[["split_id", "donor_id", "y_true"]]
assert observed_test_rows.merge(
    expected_test_rows, on=["split_id", "donor_id", "y_true"],
    validate="one_to_one",
).shape[0] == len(expected_test_rows) == len(observed_test_rows)
assert cellcnn_selection["gate"].eq(GATE).all()
assert cellcnn_selection["run_mode"].eq(RUN_MODE).all()
assert set(cellcnn_selection["device"]).issubset({"cpu", "cuda"})
assert cellcnn_selection["implementation_version"].eq(IMPLEMENTATION_VERSION).all()
assert cellcnn_filters["gate"].eq(GATE).all()
assert cellcnn_filters["run_mode"].eq(RUN_MODE).all()
assert set(cellcnn_filters["device"]).issubset({"cpu", "cuda"})
assert cellcnn_filters["implementation_version"].eq(IMPLEMENTATION_VERSION).all()
assert cellcnn_filters["cofactor"].eq(COFACTOR).all()
assert cellcnn_filters["top_fraction"].eq(TOP_FRACTION).all()

display(cellcnn_selection)
display(cellcnn_predictions)

Training deaktiviert; vorhandene CellCNN-Ergebnisse werden geladen.


,split_id,gate,run_mode,device,implementation_version,inner_fold,filter_count,candidate_seed,validation_accuracy,validation_roc_auc,best_validation_loss,epochs_run,selected
0,0,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3,3003105992,0.60,0.666667,0.635431,10,False
1,0,gated_NK,smoke,cuda,pytorch_materialized_v1,0,5,3003106192,0.60,0.666667,0.643744,7,False
2,0,gated_NK,smoke,cuda,pytorch_materialized_v1,1,3,3003115992,0.60,1.000000,0.473219,12,False
3,0,gated_NK,smoke,cuda,pytorch_materialized_v1,1,5,3003116192,0.80,0.833333,0.540527,12,True
4,0,gated_NK,smoke,cuda,pytorch_materialized_v1,2,3,3003125992,0.75,1.000000,0.418682,12,False
5,0,gated_NK,smoke,cuda,pytorch_materialized_v1,2,5,3003126192,0.75,1.000000,0.306281,12,False
6,1,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3,976401080,0.60,0.500000,0.683635,4,False
7,1,gated_NK,smoke,cuda,pytorch_materialized_v1,0,5,976401280,0.80,0.833333,0.490548,12,False
8,1,gated_NK,smoke,cuda,pytorch_materialized_v1,1,3,976411080,0.60,0.500000,0.676885,5,False
9,1,gated_NK,smoke,cuda,pytorch_materialized_v1,1,5,976411280,0.60,0.666667,0.697009,10,False


,method,gate,run_mode,device,implementation_version,split_id,split_seed,donor_id,y_true,score,decision_threshold,y_pred,filter_count,selected_inner_fold,training_cells_per_input,training_inputs_per_donor,prediction_cells_per_input,prediction_inputs_per_donor,top_fraction
0,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3003105692,a_002,1,0.567830,0.5,1,5,1,1000,8,3000,5,0.01
1,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3003105692,a_004,0,0.123402,0.5,0,5,1,1000,8,3000,5,0.01
2,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3003105692,a_006,0,0.471857,0.5,0,5,1,1000,8,3000,5,0.01
3,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3003105692,a_1a,0,0.441306,0.5,0,5,1,1000,8,3000,5,0.01
4,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3003105692,a_2a,0,0.330092,0.5,0,5,1,1000,8,3000,5,0.01
5,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,0,3003105692,a_4a,1,0.841874,0.5,1,5,1,1000,8,3000,5,0.01
6,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,1,976400780,a_002,1,0.730623,0.5,1,5,2,1000,8,3000,5,0.01
7,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,1,976400780,a_003,0,0.827457,0.5,1,5,2,1000,8,3000,5,0.01
8,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,1,976400780,a_004,0,0.277516,0.5,0,5,2,1000,8,3000,5,0.01
9,cellcnn,gated_NK,smoke,cuda,pytorch_materialized_v1,1,976400780,a_006,0,0.553730,0.5,1,5,2,1000,8,3000,5,0.01


## 7. Spenderbezogene Smoke-Test-Metriken

In [7]:
def trapezoidal_pr_auc(y_true, scores):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    return float(auc(recall, precision))

cellcnn_metrics = (
    cellcnn_predictions.groupby("split_id")
    .apply(
        lambda group: pd.Series(
            {
                "roc_auc": roc_auc_score(group["y_true"], group["score"]),
                "pr_auc": trapezoidal_pr_auc(group["y_true"], group["score"]),
                "balanced_accuracy": balanced_accuracy_score(
                    group["y_true"], group["y_pred"]
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
display(cellcnn_metrics)

,split_id,roc_auc,pr_auc,balanced_accuracy
0,0,1.000,1.000000,1.00
1,1,0.875,0.791667,0.75


## Abschluss von Schritt 4

CellCNN ist in PyTorch implementiert und zunächst isoliert im kleinen deterministischen Modus geprüft. Neben Spendervorhersagen werden die ausgewählten Filtergewichte, Ausgabekontraste und Skalierungsparameter gespeichert, damit die Zellantworten für Aufgabe 5 reproduzierbar berechnet werden können. Smoke-Test-Metriken sind nicht für den Bericht bestimmt.